# Task 7 — TimesNet → SSA smoke test

Run top to bottom from the PTDLKD project root in the Task 3 Python environment. The notebook clones the author baseline at the Task 8 reference commit and applies the local smoke-path patch. It filters both official splits using the same deterministic list of 100 store–product pairs, then runs raw SSA, TimesNet recovery, and SSA on recovered demand. All outputs are smoke results and must not be copied into Table 3.


In [ ]:
from pathlib import Path
import os, sys, subprocess, pandas as pd, numpy as np
ROOT = Path.cwd()
REPO = ROOT / 'frn-50k-baseline'
OUT = ROOT / 'Task07_Smoke_Test'
SMOKE = OUT / 'smoke'
SMOKE.mkdir(parents=True, exist_ok=True)
DATA = REPO / 'latent_demand_recovery' / 'exp' / 'demand'
DATA.mkdir(parents=True, exist_ok=True)
TRAIN = DATA / 'train_smoke.parquet'
EVAL = DATA / 'eval_smoke.parquet'
PAIRS = OUT / 'smoke_pairs.csv'
PRED = OUT / 'demand_timesnet_smoke.parquet'
print('Python:', sys.executable, '| Repo:', REPO, '| Outputs:', OUT)
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Dingdong-Inc/frn-50k-baseline.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '8af694a09a5a427d1d50648d348c022565a781f2'], check=True)
patch_script = OUT / 'patch_baseline.py'
subprocess.run([sys.executable, str(patch_script), str(REPO)], check=True)

## Select and persist 100 complete pairs
Require exactly 90 train rows and 7 eval rows per pair, at least one train stockout, and at least one eval row with stock_hour6_22_cnt == 0. Sampling is deterministic.

In [ ]:
from datasets import load_dataset
ds = load_dataset('Dingdong-Inc/FreshRetailNet-50K')
train_all, eval_all = ds['train'].to_pandas(), ds['eval'].to_pandas()
keys = ['store_id', 'product_id']
train_n = train_all.groupby(keys).size()
eval_n = eval_all.groupby(keys).size()
train_stockout = train_all.groupby(keys)['stock_hour6_22_cnt'].max().gt(0)
eval_valid = eval_all.loc[eval_all.stock_hour6_22_cnt.eq(0)].groupby(keys).size().gt(0)
eligible = train_n[train_n.eq(90)].index.intersection(eval_n[eval_n.eq(7)].index)
eligible = eligible.intersection(train_stockout[train_stockout].index).intersection(eval_valid[eval_valid].index)
eligible_df = pd.DataFrame(eligible.tolist(), columns=keys).sample(n=100, random_state=20261116).sort_values(keys).reset_index(drop=True)
pairs = pd.MultiIndex.from_frame(eligible_df[keys])
train_smoke = train_all.set_index(keys).loc[train_all.set_index(keys).index.isin(pairs)].reset_index().sort_values(keys+['dt']).reset_index(drop=True)
eval_smoke = eval_all.set_index(keys).loc[eval_all.set_index(keys).index.isin(pairs)].reset_index().sort_values(keys+['dt']).reset_index(drop=True)
assert eligible_df.drop_duplicates().shape[0] == 100
assert len(train_smoke) == 9000 and len(eval_smoke) == 700, (len(train_smoke), len(eval_smoke))
assert train_smoke.groupby(keys).size().eq(90).all() and eval_smoke.groupby(keys).size().eq(7).all()
eligible_df.to_csv(PAIRS, index=False)
train_smoke.to_parquet(TRAIN, index=False)
eval_smoke.to_parquet(EVAL, index=False)
print(f'eligible={len(eligible)}; pairs={len(eligible_df)}; train={len(train_smoke)}; eval={len(eval_smoke)}')

## Run SSA on raw sales
This passes the same filtered train and eval Parquets to the forecasting script.

In [ ]:
ssa = REPO / 'demand_forecasting' / 'SSA' / 'ssa_forecasting.py'
raw_metrics = OUT / 'ssa_raw_metrics_smoke.csv'
with open(SMOKE / 'ssa_raw.log', 'w', encoding='utf-8') as log:
    subprocess.run([sys.executable, str(ssa), '--train_path', str(TRAIN), '--eval_path', str(EVAL), '--metrics_path', str(raw_metrics)], cwd=ssa.parent, stdout=log, stderr=subprocess.STDOUT, check=True)
print((SMOKE / 'ssa_raw.log').read_text(encoding='utf-8')[-2000:])

## Run TimesNet recovery on the filtered train set
Smoke setting: `--missing_rate 0 --epochs 1` (reduced from the repository default of 5 epochs). Both `load_data` and `_demand_recovery` receive the same filtered Parquet. The prediction output is named `demand_timesnet_smoke.parquet`.

In [ ]:
app = REPO / 'latent_demand_recovery' / 'exp' / 'app.py'
with open(SMOKE / 'timesnet.log', 'w', encoding='utf-8') as log:
    subprocess.run([sys.executable, str(app), '--model', 'TimesNet', '--missing_rate', '0', '--epochs', '1', '--train_path', str(TRAIN), '--output_path', str(PRED)], cwd=app.parent, stdout=log, stderr=subprocess.STDOUT, check=True)
print((SMOKE / 'timesnet.log').read_text(encoding='utf-8')[-3000:])

In [ ]:
pred = pd.read_parquet(PRED)
assert len(pred) == 9000, len(pred)
assert pred[['store_id','product_id','dt']].equals(train_smoke[['store_id','product_id','dt']])
vals = pred.sale_amount_pred.to_numpy(dtype=float)
assert np.isfinite(vals).all() and (vals >= 0).all(), 'Recovered predictions must be finite and nonnegative.'
assert pred.groupby(keys).size().eq(90).all()
print('Validated 9,000 prediction rows; all keys match; sale_amount_pred finite and nonnegative.')

## Run SSA on TimesNet demand and write smoke metrics
The eval split is the same 700 rows used by raw SSA. WAPE/WPE retain the baseline's official daily aggregation; exported values are percentage points.

In [ ]:
demand_metrics = OUT / 'ssa_timesnet_metrics_smoke.csv'
with open(SMOKE / 'ssa_timesnet.log', 'w', encoding='utf-8') as log:
    subprocess.run([sys.executable, str(ssa), '--demand', '--demand_path', str(PRED), '--eval_path', str(EVAL), '--metrics_path', str(demand_metrics)], cwd=ssa.parent, stdout=log, stderr=subprocess.STDOUT, check=True)
raw = pd.read_csv(raw_metrics).assign(config_id='SSA_RawSales')
recovered = pd.read_csv(demand_metrics).assign(config_id='SSA_TimesNet')
metrics = pd.concat([raw, recovered], ignore_index=True)
metrics = metrics.rename(columns={'wape':'wape_fraction','wpe':'wpe_fraction'})
metrics['group'] = metrics.group.map({'psd>=0':'Overall','psd<1':'Low Sale','psd>=1':'High Sale'}).fillna(metrics.group)
metrics['wape_pct'] = metrics.wape_fraction * 100
metrics['wpe_pct'] = metrics.wpe_fraction * 100
psd = train_smoke.groupby(keys)['sale_amount'].mean().rename('psd').reset_index()
eval_eligible = eval_smoke.loc[eval_smoke.stock_hour6_22_cnt.eq(0)].merge(psd, on=keys, how='left')
valid_counts = {
    'Overall': len(eval_eligible),
    'Low Sale': int((eval_eligible.psd < 1).sum()),
    'High Sale': int((eval_eligible.psd >= 1).sum()),
}
metrics['valid_eval_rows'] = metrics.group.map(valid_counts)
finite = np.isfinite(metrics.wape_pct) & np.isfinite(metrics.wpe_pct)
metrics['status'] = np.where(metrics.valid_eval_rows.eq(0), 'no_valid_rows',
                             np.where(finite, 'ok', 'no_finite_metrics'))
metrics[['config_id','group','wape_pct','wpe_pct','status']].to_csv(OUT / 'smoke_metrics.csv', index=False)
overall = metrics.query("group == 'Overall'")
assert len(overall) == 2 and overall.status.eq('ok').all(), overall
print(metrics[['config_id','group','wape_pct','wpe_pct','status']].to_string(index=False))
print('Both Overall branches completed with finite WAPE/WPE.')

In [ ]:
info = f'''# Task 7 smoke run

Completed at runtime: {pd.Timestamp.now()}

- Pair seed: 20261116; selected pairs: {len(eligible_df)}
- Train rows: {len(train_smoke)}; eval rows: {len(eval_smoke)}
- Eligibility: exactly 90 train rows and 7 eval rows per pair; >=1 train stockout; >=1 eval row with stock_hour6_22_cnt == 0.
- TimesNet: model=TimesNet, missing_rate=0, epochs=1 (smoke reduction from default 5), seed=1024 from baseline.
- SSA raw and recovered demand: both used the same filtered train/eval files.
- Outputs are smoke results and must not be copied into Table 3.
'''
(OUT / 'run_info.md').write_text(info, encoding='utf-8')
print('Wrote run_info.md; Task 7 smoke artifacts are complete.')